<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/05_medical_baseline_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Tunix-Med · Part 1: Establishing the Baseline

In this notebook, we measure the "raw" medical knowledge of the **Gemma 3 270M** model before any fine-tuning. This is a critical step in the domain adaptation process: without a robust baseline, we cannot honestly quantify the value of our training.

### Why this matters for the Workshop
1.  **Benchmarking Philosophy**: We use a small model (270M) to demonstrate that even a "clumsy" model can be transformed into a specialist.
2.  **Metric Integrity**: We introduce three independent ways to measure truth—Keyword overlap, Semantic alignment, and AI Judging.
3.  **Zero-Shot Evaluation**: We test the model's ability to follow a clinical persona without having seen our specific training pairs yet.

### The Evaluation Pipeline
1.  **Model Loading**: Loading the base model in `bfloat16` for memory efficiency.
2.  **Dataset Reconstruction**: Sampling 300 questions from the held-out split of our cardiology dataset.
3.  **Inference**: Generating answers using a consistent Clinical System Prompt.
4.  **Scoring**: Calculating the baseline scores that we will later compare against the fine-tuned model.

## 1 · Environment Setup & Model Loading

We start by configuring the environment. We use `bfloat16` if the GPU supports it (Ampere architecture or newer like A100/H100), as it prevents numerical overflows that common `float16` can suffer from with Gemma models.

In [1]:
import os
import warnings
import logging
import re
import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import login

# Silence unnecessary logs
warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Authenticate with Hugging Face to access the model
token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)


def info_device():
    """Detect the best available hardware accelerator."""
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


device = info_device()

# Gemma 3 performs best in bfloat16 on Ampere+ GPUs (RTX 30/40, A100, etc.)
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float32
)
print(f"Device : {device}  |  dtype : {dtype}")

# We use a Gemma 270M Instruction Tuned version as our starting point
MODEL_NAME = "google/gemma-3-270m-it"
print(f"Loading base model: {MODEL_NAME} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map=device,
)
model.eval()
print("Model ready for baseline inference.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device : cuda  |  dtype : torch.bfloat16
Loading base model: google/gemma-3-270m-it ...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model ready for baseline inference.


## 2 · Reproducible Evaluation Split

To ensure a fair comparison, we must evaluate the model on data it has **not** seen during training. We use a 10% held-out split and a fixed random seed (`42`). This same split will be used in the final evaluation notebook.

In [ ]:
DATASET_ID = "lmassaron/medical-cardiology-qa"
EVAL_SPLIT = 0.1
SEED = 42
N_EVAL_QS = 100

print(f"Loading {DATASET_ID} ...")
full_ds = load_dataset(DATASET_ID, split="train")
n = len(full_ds)

# Create reproducible indices for the eval split
rng = np.random.default_rng(SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - EVAL_SPLIT))
eval_idx = all_idx[cut:]

print(f"  Total rows : {n:,}  |  Eval rows : {len(eval_idx):,}")


def extract_qa(example):
    """Extract user and assistant messages from the chat format."""
    msgs = example["messages"]
    return {
        "question": next(m["content"] for m in msgs if m["role"] == "user"),
        "answer": next(m["content"] for m in msgs if m["role"] == "assistant"),
    }


# Shuffle the eval set and sample N questions
rng2 = np.random.default_rng(SEED + 1)
shuffled = rng2.permutation(eval_idx)

seen_prefixes, qa_pairs = set(), []
for idx in shuffled:
    if len(qa_pairs) >= N_EVAL_QS:
        break
    ex = extract_qa(full_ds[int(idx)])
    q, a = ex["question"], ex["answer"]

    # Filtering for quality: avoid very short answers and redundant questions
    if len(a) < 25:
        continue
    prefix = " ".join(q.lower().split()[:4])
    if prefix in seen_prefixes:
        continue

    seen_prefixes.add(prefix)
    qa_pairs.append({"question": q, "answer": a, "dataset_idx": int(idx)})

data = pd.DataFrame(qa_pairs)
print(f"\nSampled {len(data)} evaluation questions.")

Loading lmassaron/medical-cardiology-qa ...
  Total rows : 10,518  |  Eval rows : 1,052

Sampled 30 evaluation questions.


## 3 · Inference Loop

We define a **System Prompt** that gives the model its clinical persona. It is vital that this prompt is identical to the one used during SFT training, otherwise, we are testing the prompt's effect rather than the training's effect.

In [3]:
# The Clinical Assistant Persona
SYSTEM_PROMPT = (
    "You are a knowledgeable medical assistant specializing in cardiology. "
    "Answer clinical questions accurately, focusing on diagnostic criteria, "
    "treatment guidelines, and pathophysiology."
)

results_list = []
model.eval()

for _, row in tqdm(data.iterrows(), total=len(data)):
    question = row["question"]
    answer = row["answer"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # Apply chat template for consistent formatting
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded,
            max_new_tokens=256,
            do_sample=False,  # Deterministic greedy decoding
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = tokenizer.decode(
        outputs[0, encoded["input_ids"].shape[-1] :], skip_special_tokens=True
    ).strip()

    # Calculate Perplexity: measures how 'surprised' the model is by the ground truth answer
    ref_ids = tokenizer(
        answer, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = torch.cat([encoded["input_ids"], ref_ids], dim=1)
    labels = full_ids.clone()
    labels[
        :, : encoded["input_ids"].shape[1]
    ] = -100  # Only calculate loss on the answer part

    with torch.no_grad():
        loss = model(full_ids, labels=labels).loss
    perplexity = torch.exp(loss).item()

    results_list.append(
        {
            "question": question,
            "expected_answer": answer,
            "generated_answer": generated,
            "perplexity": perplexity,
        }
    )

results_df = pd.DataFrame(results_list)
print(f"Generated {len(results_df)} answers.")

  0%|          | 0/30 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated 30 answers.


## 4 · Scoring Metrics Logic

We use a composite scoring strategy to capture different aspects of model performance:
1.  **Keyword F1 (TF-IDF)**: Rewards correctly using domain-specific terms (e.g., "neprilysin") more than common words.
2.  **Semantic Similarity**: Uses Sentence-BERT to check if the *meaning* is preserved even if phrasings differ.
3.  **AI Judge**: Uses a larger, smarter model (**Qwen 2.5 7B**) to fact-check the answer through reasoning (Chain-of-Thought).

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Keyword F1 with TF-IDF Weights
# ---------------------------------
# We fit TF-IDF on the whole eval set to identify rare, important medical words.
_tfidf = TfidfVectorizer(
    analyzer="word", token_pattern=r"\b\w{4,}\b", sublinear_tf=True
)
_tfidf.fit(results_df["expected_answer"].tolist())
_vocab = _tfidf.vocabulary_
_idf = _tfidf.idf_


def keyword_f1_tfidf(generated: str, expected: str) -> float:
    ref_kws = set(re.findall(r"\b\w{4,}\b", expected.lower()))
    gen_kws = set(re.findall(r"\b\w{4,}\b", generated.lower()))
    if not ref_kws:
        return 1.0

    def weighted_count(kws, universe):
        return sum(
            _idf[_vocab[w]] if w in _vocab else 1.0 for w in universe if w in kws
        )

    ref_weight = sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in ref_kws)
    gen_weight = (
        sum(_idf[_vocab[w]] if w in _vocab else 1.0 for w in gen_kws)
        if gen_kws
        else 0.0
    )

    if ref_weight == 0 or gen_weight == 0:
        return 0.0
    recall = weighted_count(gen_kws, ref_kws) / ref_weight
    precision = weighted_count(ref_kws, gen_kws) / gen_weight
    return float(
        (2 * precision * recall / (precision + recall))
        if (precision + recall) > 0
        else 0.0
    )

In [5]:
# 2. Semantic Similarity Setup
# ---------------------------
sim_model = SentenceTransformer("all-MiniLM-L6-v2")


def _raw_semantic(generated: str, expected: str) -> float:
    e1 = sim_model.encode(generated, convert_to_tensor=True, show_progress_bar=False)
    e2 = sim_model.encode(expected, convert_to_tensor=True, show_progress_bar=False)
    return float(util.pytorch_cos_sim(e1, e2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# 3. AI Judge Setup (Qwen 7B in 4-bit)
# -----------------------------------
from transformers import BitsAndBytesConfig

JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

judge_mdl = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, quantization_config=bnb_config, device_map="auto"
)


def ai_judge(question: str, generated: str, expected: str) -> float:
    prompt = (
        "Evaluate factuality of the generated answer against the reference.\n"
        f"Q: {question}\nRef: {expected}\nGen: {generated}\n"
        "First write reasoning, then on the last line write ONLY the score 1-10."
    )
    inp = judge_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(judge_mdl.device)

    with torch.no_grad():
        out = judge_mdl.generate(**inp, max_new_tokens=120, do_sample=False)

    txt = judge_tok.decode(
        out[0, inp["input_ids"].shape[-1] :], skip_special_tokens=True
    ).strip()
    m = re.search(r"\b(\d+)\b", txt.splitlines()[-1])
    return max(min((int(m.group(1)) / 10.0) if m else 0.5, 1.0), 0.1)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 5 · Final Calculations & Export

We calculate the mean scores. Note the **Calibration** step: raw similarity scores usually cluster in the 0.4-0.9 range. We stretch this to the full 0-1 range based on the baseline min/max to make the final score more interpretable.

In [11]:
results_df["keyword_score"] = results_df.apply(
    lambda r: keyword_f1_tfidf(r["generated_answer"], r["expected_answer"]), axis=1
)
results_df["_raw_sim"] = results_df.apply(
    lambda r: _raw_semantic(r["generated_answer"], r["expected_answer"]), axis=1
)

sim_min, sim_max = results_df["_raw_sim"].min(), results_df["_raw_sim"].max()
results_df["semantic_score"] = (
    (results_df["_raw_sim"] - sim_min) / (sim_max - sim_min)
).clip(0, 1)

print("Running AI Judge (this may take a few minutes) ...")
scores = []
for _, r in tqdm(results_df.iterrows(), total=len(results_df), desc="AI Judge"):
    score = ai_judge(r["question"], r["generated_answer"], r["expected_answer"])
    scores.append(score)
results_df["ai_judge_score"] = scores

results_df["final_score"] = (
    results_df["keyword_score"] * 0.1
    + results_df["semantic_score"] * 0.3
    + results_df["ai_judge_score"] * 0.6
)

print("\n--- BASELINE RESULTS ---")
print(f"  Mean Keyword Score  : {results_df['keyword_score'].mean():.3f}")
print(f"  Mean Semantic Score : {results_df['semantic_score'].mean():.3f}")
print(f"  Mean AI Judge Score : {results_df['ai_judge_score'].mean():.3f}")
print(f"  Mean Final Score    : {results_df['final_score'].mean():.3f}")
print(f"  Mean Perplexity     : {results_df['perplexity'].mean():.1f}")
print("------------------------")

results_df.to_csv("medical_baseline_results.csv", index=False)
print("Saved baseline to medical_baseline_results.csv")

Running AI Judge (this may take a few minutes) ...


AI Judge:   0%|          | 0/30 [00:00<?, ?it/s]


--- BASELINE RESULTS ---
  Mean Keyword Score  : 0.156
  Mean Semantic Score : 0.588
  Mean AI Judge Score : 0.633
  Mean Final Score    : 0.572
  Mean Perplexity     : 2780.5
------------------------
Saved baseline to medical_baseline_results.csv
